# Notebook 5 (per-view, crash-safe): prune + train + validate

The per-view projects are **built and labeled in Notebook 4** (create projects, extract frames, label only
what each view sees). This notebook takes those projects and trains them, fully re-runnable:

- **No migration / no bodypart overwrites**, the per-view projects' own `labeled-data/` is the single
  source of truth. The old §4/§5 (restrict-from-pooled, migrate) are gone. They could clobber hand edits.
- **Prune** each project's `bodyparts` to what was actually labeled (keeps any keypoint you added by hand).
- **Guarded** dataset creation and training, skips anything already on disk, so a crash never costs a run.
  Flip `FORCE_REBUILD` / `FORCE_RETRAIN` after (re)labeling.
- **Validation** runs the trained checkpoint on a short clip and renders an overlay video (libx264), it
  never calls `evaluate_network(plotting=True)`, which is what reset the GPU.

If a view's project is missing, build it in Notebook 4 first.

## 0. Imports, paths, discover the projects

In [1]:
import deeplabcut as dlc
from deeplabcut.core.engine import Engine
from pathlib import Path
import shutil, yaml, numpy as np, pandas as pd, torch, cv2

OUT          = Path('../data/lockbox_dlc/scene1').resolve()
PERVIEW_ROOT = OUT / 'perview'
SCORER, SHUFFLE = 'htcv', 1
device  = 'cuda:0' if torch.cuda.is_available() else 'cpu'
VIEWS   = ['top', 'side', 'front']
print('DeepLabCut:', dlc.__version__, '| device:', device)

# ---- disk-state helpers (basis for all skip logic) ----
def find_config(view):
    h = sorted(PERVIEW_ROOT.glob(f'lockbox_{view}-{SCORER}-*/config.yaml')); return h[-1] if h else None
def train_dir(cfgp):
    h = list((cfgp.parent / 'dlc-models-pytorch').rglob(f'*shuffle{SHUFFLE}/train')); return h[0] if h else None
def has_traindata(cfgp):
    td = train_dir(cfgp); return bool(td and (td / 'pytorch_config.yaml').exists())
def list_snapshots(cfgp):
    td = train_dir(cfgp); return sorted(td.glob('snapshot-*.pt')) if td else []

PERVIEW_CONFIG = {}
for v in VIEWS:
    c = find_config(v)
    if c is None:
        print(f'{v:<6} MISSING — build it in Notebook 4'); continue
    PERVIEW_CONFIG[v] = c
    snaps = list_snapshots(c)
    snap_s = f'{len(snaps)} (latest {snaps[-1].name})' if snaps else 'none'
    print(f'{v:<6} {c.parent.name:<34} traindata={"yes" if has_traindata(c) else "no":<3} snapshots={snap_s}')

Loading DLC 3.0.0rc13...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)


/home/kenny/HTCV/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DeepLabCut: 3.0.0rc13 | device: cuda:0
top    lockbox_top-htcv-2026-05-25        traindata=yes snapshots=6 (latest snapshot-best-120.pt)
side   lockbox_side-htcv-2026-05-25       traindata=yes snapshots=6 (latest snapshot-best-130.pt)
front  lockbox_front-htcv-2026-05-25      traindata=yes snapshots=6 (latest snapshot-best-120.pt)


## 1. Prune each project's bodyparts to what was labeled

Safe and idempotent: it only **reads** each project's labels and **writes** its `config.yaml` bodyparts
it never touches the label files. A keypoint labeled in `>= MIN_LABELED` frames is kept (so anything you
added by hand stays). The rest are dropped so the model isn't asked to predict things that view never sees.
A view with no labels yet is skipped (label it in Notebook 4 first).

In [2]:
MIN_LABELED = 3

def labeled_counts(cfgp):
    counts = {}
    for h5 in (cfgp.parent / 'labeled-data').rglob(f'CollectedData_{SCORER}.h5'):
        df = pd.read_hdf(h5)
        for bp in df.columns.get_level_values(-2).unique():
            counts[bp] = counts.get(bp, 0) + int(df.loc[:, (slice(None), bp, 'x')].notna().sum().sum())
    return counts

READY = []
for view, cfgp in PERVIEW_CONFIG.items():
    c = labeled_counts(cfgp)
    keep = [bp for bp in sorted(c) if c[bp] >= MIN_LABELED]
    if not keep:
        print(f'{view:<6} no labels yet — label it in Notebook 4 (skipping)'); continue
    cfg = yaml.safe_load(open(cfgp)); cfg['bodyparts'] = keep; cfg['numframes2pick'] = 0
    yaml.safe_dump(cfg, open(cfgp, 'w'), sort_keys=False)
    READY.append(view)
    print(f'\n{view:<6} bodyparts -> {keep}')
    for bp in sorted(c):
        print(f'      {"✓" if c[bp] >= MIN_LABELED else "·"} {bp:<22} {c[bp]:>4} labeled')

print(f'\nready to train: {READY}')


top    bodyparts -> ['ball1_center', 'ball1_cradle', 'cover_guidance_left', 'cover_guidance_right', 'cover_marker', 'lever1_push_end', 'slider1_hole_lockbox', 'slider1_knob', 'slider1_stabilizer']
      ✓ ball1_center            532 labeled
      ✓ ball1_cradle             26 labeled
      ✓ cover_guidance_left     590 labeled
      ✓ cover_guidance_right    423 labeled
      ✓ cover_marker            432 labeled
      ✓ lever1_push_end         662 labeled
      ✓ slider1_hole_lockbox    505 labeled
      ✓ slider1_knob            659 labeled
      ✓ slider1_stabilizer      656 labeled

side   bodyparts -> ['ball1_center', 'cover_guidance_left', 'cover_guidance_right', 'cover_marker', 'lever1_pivot', 'lever1_push_end', 'slider1_hole_lockbox', 'slider1_knob', 'slider1_stabilizer']
      ✓ ball1_center            456 labeled
      · ball1_cradle              1 labeled
      ✓ cover_guidance_left     248 labeled
      ✓ cover_guidance_right     90 labeled
      ✓ cover_marker            

## 2. Create training datasets + train (guarded)

Both steps skip what already exists, so re-running after a crash retrains nothing.

- After **(re)labeling or pruning changed the bodyparts**, set `FORCE_REBUILD = True` once so the training
  dataset (and the model head) pick up the change. Adding a keypoint changes the head, so the old snapshots
  can't be resumed, `FORCE_REBUILD` removes the stale shuffle and a fresh train follows.
- `FORCE_RETRAIN = True` forces retraining even when snapshots exist.

In [ ]:
NET_TYPE = 'resnet_50'
EPOCHS, SAVE_EPOCHS, BATCH_SIZE = 200, 25, 2
# Reuse the already-trained per-view models on this rerun (all three views have snapshots on disk).
FORCE_VIEWS = set()


for view in READY:
    cfgp  = PERVIEW_CONFIG[view]
    force = view in FORCE_VIEWS

    # --- training dataset ---
    if has_traindata(cfgp) and not force:
        print(f'{view:<6} dataset exists — skip')
    else:
        if force:
            for sub in ('dlc-models-pytorch', 'training-datasets'):
                d = cfgp.parent / sub
                if d.exists():
                    shutil.rmtree(d)
            print(f'{view:<6} cleared derived artifacts (forced rebuild)')
        dlc.create_training_dataset(str(cfgp), num_shuffles=1, net_type=NET_TYPE, engine=Engine.PYTORCH)
        print(f'{view:<6} created training dataset')

    # --- train ---
    if list_snapshots(cfgp) and not force:
        print(f'{view:<6} trained ({len(list_snapshots(cfgp))} snapshots) — skip')
        continue
    print(f'=== training {view} ===')
    dlc.train_network(str(cfgp), shuffle=SHUFFLE, device=device,
                      epochs=EPOCHS, save_epochs=SAVE_EPOCHS, batch_size=BATCH_SIZE)

top    dataset exists — skip
top    trained (6 snapshots) — skip
side   dataset exists — skip
side   trained (6 snapshots) — skip
front  dataset exists — skip
front  trained (6 snapshots) — skip


## 3. Crash-safe validation (overlay video)

Cuts a short clip from the view's video, runs the **trained checkpoint** on it (via `analyze_videos`,
proven safe on this GPU), and renders predictions onto the original frames as a normal H.264 mp4. Idempotent
(clip + analysis are reused if present), and it never touches `evaluate_network(plotting=True)`.

In [ ]:
VAL_VIEW = 'top'
assert VAL_VIEW in READY and list_snapshots(PERVIEW_CONFIG[VAL_VIEW]), \
    f'{VAL_VIEW}: not trained yet'
cfgp = PERVIEW_CONFIG[VAL_VIEW]

# source video is registered in the config (projects use copy_videos=False)
cfg = yaml.safe_load(open(cfgp)); cfg['snapshotindex'] = -1
yaml.safe_dump(cfg, open(cfgp, 'w'), sort_keys=False)
vid = Path(list(cfg['video_sets'].keys())[0])

cap = cv2.VideoCapture(str(vid))
FPS = cap.get(cv2.CAP_PROP_FPS) or 30
TOT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

START_S, DUR_S = 170, 210            # pick a window
f0 = int(START_S * FPS); n = min(int(DUR_S * FPS), TOT - f0)

evdir = cfgp.parent / 'eval_tmp'; evdir.mkdir(exist_ok=True)
clip  = evdir / f'{VAL_VIEW}_clip_{START_S}s_{DUR_S}s.avi'
if not (clip.exists() and clip.stat().st_size > 0):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
    wr = None
    for cc in ('FFV1', 'HFYU'):
        cand = cv2.VideoWriter(str(clip), cv2.VideoWriter_fourcc(*cc), FPS, (W, H))
        if cand.isOpened(): wr = cand; break
    if wr is None: raise RuntimeError('no lossless VideoWriter codec available')
    for _ in range(n):
        ok, fr = cap.read()
        if not ok: break
        wr.write(fr)
    wr.release(); print(f'wrote clip {clip.name} ({n} frames)')
else:
    print(f'{clip.name} exists — skip encoding')
cap.release()

clip_pred = next(iter(evdir.glob(f'{clip.stem}*.h5')), None)
if clip_pred is None:
    dlc.analyze_videos(str(cfgp), [str(clip)], shuffle=SHUFFLE, device=device, save_as_csv=False)
    clip_pred = next(evdir.glob(f'{clip.stem}*.h5'))
else:
    print(f'{clip_pred.name} exists — skip inference')

top_clip_170s_210s.avi exists — skip encoding
top_clip_170s_210sDLC_Resnet50_lockbox_topMay25shuffle1_snapshot_best-120.h5 exists — skip inference


In [ ]:
VAL_VIEW = 'side'
assert VAL_VIEW in READY and list_snapshots(PERVIEW_CONFIG[VAL_VIEW]), \
    f'{VAL_VIEW}: not trained yet'
cfgp = PERVIEW_CONFIG[VAL_VIEW]

# source video is registered in the config (projects use copy_videos=False)
cfg = yaml.safe_load(open(cfgp)); cfg['snapshotindex'] = -1
yaml.safe_dump(cfg, open(cfgp, 'w'), sort_keys=False)
vid = Path(list(cfg['video_sets'].keys())[0])

cap = cv2.VideoCapture(str(vid))
FPS = cap.get(cv2.CAP_PROP_FPS) or 30
TOT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

START_S, DUR_S = 170, 210            # pick a window
f0 = int(START_S * FPS); n = min(int(DUR_S * FPS), TOT - f0)

evdir = cfgp.parent / 'eval_tmp'; evdir.mkdir(exist_ok=True)
clip  = evdir / f'{VAL_VIEW}_clip_{START_S}s_{DUR_S}s.avi'
if not (clip.exists() and clip.stat().st_size > 0):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
    wr = None
    for cc in ('FFV1', 'HFYU'):
        cand = cv2.VideoWriter(str(clip), cv2.VideoWriter_fourcc(*cc), FPS, (W, H))
        if cand.isOpened(): wr = cand; break
    if wr is None: raise RuntimeError('no lossless VideoWriter codec available')
    for _ in range(n):
        ok, fr = cap.read()
        if not ok: break
        wr.write(fr)
    wr.release(); print(f'wrote clip {clip.name} ({n} frames)')
else:
    print(f'{clip.name} exists — skip encoding')
cap.release()

clip_pred = next(iter(evdir.glob(f'{clip.stem}*.h5')), None)
if clip_pred is None:
    dlc.analyze_videos(str(cfgp), [str(clip)], shuffle=SHUFFLE, device=device, save_as_csv=False)
    clip_pred = next(evdir.glob(f'{clip.stem}*.h5'))
else:
    print(f'{clip_pred.name} exists — skip inference')

side_clip_170s_210s.avi exists — skip encoding
side_clip_170s_210sDLC_Resnet50_lockbox_sideMay25shuffle1_snapshot_best-130.h5 exists — skip inference


In [ ]:
VAL_VIEW = 'front'
assert VAL_VIEW in READY and list_snapshots(PERVIEW_CONFIG[VAL_VIEW]), \
    f'{VAL_VIEW}: not trained yet'
cfgp = PERVIEW_CONFIG[VAL_VIEW]

# source video is registered in the config (projects use copy_videos=False)
cfg = yaml.safe_load(open(cfgp)); cfg['snapshotindex'] = -1
yaml.safe_dump(cfg, open(cfgp, 'w'), sort_keys=False)
vid = Path(list(cfg['video_sets'].keys())[0])

cap = cv2.VideoCapture(str(vid))
FPS = cap.get(cv2.CAP_PROP_FPS) or 30
TOT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

START_S, DUR_S = 170, 210            # pick a window
f0 = int(START_S * FPS); n = min(int(DUR_S * FPS), TOT - f0)

evdir = cfgp.parent / 'eval_tmp'; evdir.mkdir(exist_ok=True)
clip  = evdir / f'{VAL_VIEW}_clip_{START_S}s_{DUR_S}s.avi'
if not (clip.exists() and clip.stat().st_size > 0):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
    wr = None
    for cc in ('FFV1', 'HFYU'):
        cand = cv2.VideoWriter(str(clip), cv2.VideoWriter_fourcc(*cc), FPS, (W, H))
        if cand.isOpened(): wr = cand; break
    if wr is None: raise RuntimeError('no lossless VideoWriter codec available')
    for _ in range(n):
        ok, fr = cap.read()
        if not ok: break
        wr.write(fr)
    wr.release(); print(f'wrote clip {clip.name} ({n} frames)')
else:
    print(f'{clip.name} exists — skip encoding')
cap.release()

clip_pred = next(iter(evdir.glob(f'{clip.stem}*.h5')), None)
if clip_pred is None:
    dlc.analyze_videos(str(cfgp), [str(clip)], shuffle=SHUFFLE, device=device, save_as_csv=False)
    clip_pred = next(evdir.glob(f'{clip.stem}*.h5'))
else:
    print(f'{clip_pred.name} exists — skip inference')

front_clip_170s_210s.avi exists — skip encoding
front_clip_170s_210sDLC_Resnet50_lockbox_frontMay25shuffle1_snapshot_best-120.h5 exists — skip inference


In [7]:
import imageio.v2 as imageio
import matplotlib.cm as cm

pred = pd.read_hdf(clip_pred)
while pred.columns.nlevels > 2: pred = pred.droplevel(0, axis=1)
bps = list(pred.columns.get_level_values(0).unique())

out = evdir / f'{VAL_VIEW}_overlay_{START_S}s_{DUR_S}s.mp4'

# The overlay is a validation visualization only; reuse it if already rendered.
if out.exists() and out.stat().st_size > 0:
    print(f'Overlay already present, skipping render -> {out}')
else:
    writer = imageio.get_writer(str(out), fps=FPS, codec='libx264', quality=8, macro_block_size=None)
    cap = cv2.VideoCapture(str(vid)); cap.set(cv2.CAP_PROP_POS_FRAMES, f0)   # render off the ORIGINAL video

    drawn = 0
    for fi in range(len(pred)):
        ok, fr = cap.read()
        if not ok: break
        for bp in bps:
            x = float(pred[bp]['x'].iloc[fi]); y = float(pred[bp]['y'].iloc[fi])
            l = float(pred[bp]['likelihood'].iloc[fi])
            if not (np.isfinite(x) and np.isfinite(y) and 0 <= x < W and 0 <= y < H): continue
            r, g, b, _ = cm.RdYlGn(max(0.0, min(1.0, l)))
            col = (int(b*255), int(g*255), int(r*255))
            cv2.circle(fr, (int(x), int(y)), 6, col, -1)
            cv2.circle(fr, (int(x), int(y)), 6, (0, 0, 0), 1)
            cv2.putText(fr, f'{bp[:14]} {l:.2f}', (int(x)+8, int(y)-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1, cv2.LINE_AA)
            drawn += 1
        cv2.putText(fr, f'{VAL_VIEW} f{f0+fi}', (12, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
        writer.append_data(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    writer.close(); cap.release()
    print(f'drawn {drawn} points -> {out.resolve()}\n')

# quick per-keypoint readout (NaN coords / off-frame / low confidence)
for bp in bps:
    x, y, l = pred[bp]['x'].values, pred[bp]['y'].values, pred[bp]['likelihood'].values
    print(f'  {bp:<22} finite_xy={np.isfinite(x).mean()*100:3.0f}%  '
          f'x[{np.nanmin(x):.0f},{np.nanmax(x):.0f}] y[{np.nanmin(y):.0f},{np.nanmax(y):.0f}]  '
          f'mean_lik={np.nanmean(l):.2f}')

Overlay already present, skipping render -> /home/kenny/HTCV/data/lockbox_dlc/scene1/perview/lockbox_front-htcv-2026-05-25/eval_tmp/front_overlay_170s_210s.mp4
  ball1_center           finite_xy=100%  x[697,1934] y[211,1023]  mean_lik=0.61
  cover_guidance_right   finite_xy=100%  x[1195,1873] y[363,953]  mean_lik=0.86
  lever1_pivot           finite_xy=100%  x[1111,1932] y[95,987]  mean_lik=0.86
  lever1_push_end        finite_xy=100%  x[1078,1774] y[205,878]  mean_lik=0.66
  slider1_hole_lockbox   finite_xy=100%  x[1726,1738] y[879,887]  mean_lik=0.88
  slider1_knob           finite_xy=100%  x[784,1547] y[210,881]  mean_lik=0.79
  slider1_stabilizer     finite_xy=100%  x[1190,1192] y[845,850]  mean_lik=0.86
